In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading dataset...")
df_train = pd.read_csv("twi_english_sample.csv")

# Build Vocabularies
twi_vocab = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}
for sentence in df_train['twi']:
    for word in str(sentence).lower().split():
        if word not in twi_vocab:
            twi_vocab[word] = len(twi_vocab)

eng_vocab = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}
for sentence in df_train['english']:
    for word in str(sentence).lower().split():
        if word not in eng_vocab:
            eng_vocab[word] = len(eng_vocab)

MAX_LEN = 10

def encode_sentence(sentence, vocab, max_len=MAX_LEN):
    tokens = str(sentence).lower().split()
    ids = [vocab.get(token, 1) for token in tokens]
    ids = [vocab["<BOS>"]] + ids + [vocab["<EOS>"]]
    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids

class TwiEnglishDataset(Dataset):
    def __init__(self, dataframe):
        self.src_sentences = dataframe['twi'].values
        self.tgt_sentences = dataframe['english'].values
        
    def __len__(self):
        return len(self.src_sentences)
        
    def __getitem__(self, idx):
        src_ids = encode_sentence(self.src_sentences[idx], twi_vocab)
        tgt_ids = encode_sentence(self.tgt_sentences[idx], eng_vocab)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

dataset = TwiEnglishDataset(df_train)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)


# ==========================================
# 2. MODEL ARCHITECTURE (Encoder, Decoder, Seq2Seq)
# ==========================================
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        
    def forward(self, src):
        embedded = self.embedding(src)
        _, hidden = self.gru(embedded)
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_vocab_size, embedding_dim)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_vocab_size)
        
    def forward(self, input_token, hidden):
        embedded = self.embedding(input_token)
        output, hidden = self.gru(embedded, hidden)
        prediction = self.fc_out(output)
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, tgt):
        hidden = self.encoder(src)
        batch_size = tgt.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.fc_out.out_features
        
        outputs = torch.zeros(batch_size, tgt_len, vocab_size)
        decoder_input = tgt[:, 0].unsqueeze(1)
        
        for t in range(1, tgt_len):
            decoder_output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = decoder_output.squeeze(1)
            decoder_input = tgt[:, t].unsqueeze(1)
            
        return outputs


# ==========================================
# 3. TRAINING & CHECKPOINTING
# ==========================================
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
LEARNING_RATE = 0.003
EPOCHS = 25

encoder_net = Encoder(vocab_size=len(twi_vocab), embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM)
decoder_net = Decoder(output_vocab_size=len(eng_vocab), embedding_dim=EMBEDDING_DIM, hidden_dim=HIDDEN_DIM)
translator_model = Seq2Seq(encoder_net, decoder_net)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(translator_model.parameters(), lr=LEARNING_RATE)

print(f"\nStarting Training Run ({EPOCHS} Epochs)...")
translator_model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for src_batch, tgt_batch in train_loader:
        optimizer.zero_grad()
        output = translator_model(src_batch, tgt_batch)
        
        output_dim = output.shape[-1]
        output_flattened = output[:, 1:].reshape(-1, output_dim)
        target_flattened = tgt_batch[:, 1:].reshape(-1)
        
        loss = criterion(output_flattened, target_flattened)
        loss.backward()
        
        # Gradient clipping for RNN/GRU stability
        torch.nn.utils.clip_grad_norm_(translator_model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Average Training Loss: {avg_loss:.4f}")

# Save weights
torch.save({
    'model_state_dict': translator_model.state_dict(),
    'twi_vocab': twi_vocab,
    'eng_vocab': eng_vocab
}, "twi_english_translator_weights.pth")
print("\nTraining complete! Weights saved as 'twi_english_translator_weights.pth'.")


# ==========================================
# 4. INFERENCE & INTERACTIVE TESTER
# ==========================================
inv_eng_vocab = {idx: word for word, idx in eng_vocab.items()}

def translate_twi_sentence(sentence, model, twi_vocab, eng_vocab, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src_ids = encode_sentence(sentence, twi_vocab, max_len)
        src_tensor = torch.tensor([src_ids], dtype=torch.long)
        hidden = model.encoder(src_tensor)
        
        decoder_input = torch.tensor([[eng_vocab["<BOS>"]]], dtype=torch.long)
        translated_words = []
        
        for _ in range(max_len):
            decoder_output, hidden = model.decoder(decoder_input, hidden)
            pred_token_id = decoder_output.argmax(dim=-1).item()
            
            if pred_token_id == eng_vocab["<EOS>"]:
                break
            word = inv_eng_vocab.get(pred_token_id, "<UNK>")
            translated_words.append(word)
            decoder_input = torch.tensor([[pred_token_id]], dtype=torch.long)
            
        return " ".join(translated_words)

# Interactive Loop
print("\n" + "=" * 50)
print("🌍 TWI-ENGLISH INTERACTIVE TRANSLATOR (Type 'exit' to quit)")
print("=" * 50)

while True:
    user_input = input("\nEnter Twi sentence: ").strip()
    if user_input.lower() == 'exit':
        print("Exiting translator. Ready for GitHub push!")
        break
    if not user_input:
        continue
    prediction = translate_twi_sentence(user_input, translator_model, twi_vocab, eng_vocab)
    print(f"-> Model Translation: {prediction}")